# **Lecture:** Tracking and video analysis

![](multi-object-tracking.jpg)

### **1. Learning Objectives**


By the end of this lab, you should be able to:

- Explain the difference between **detection** and **tracking**.
- Describe the role of **classical trackers** (KCF, CSRT) and when they are useful.
- Implement a simple **tracking-by-detection** pipeline using PyTorch and OpenCV.
- Run experiments on a **video** and observe how tracking behaves over time.
- Understand how tracking and detection relate to **real systems** like the DeepStream-based pedestrian analytics system described in the master’s thesis proposal (age and gender classification over detected people).


### **2. Detection vs Tracking**

#### 2.1 Object Detection

**Object detection** answers the question: *“What objects are in this frame, and where are they?”*.

- Input: a single image (frame).
- Output: bounding boxes + class labels (e.g., `person`).
- Typical models: Faster R-CNN, YOLO, SSD, etc.
- Cost: relatively **expensive** per frame (deep network forward pass).

#### 2.2 Object Tracking

**Object tracking** answers the question: *“Where did this object move between frames?”*.

- Input: an initial bounding box in frame *t*.
- Output: updated bounding box in frame *t+1, t+2, ...*.
- Trackers assume the object is the **same** and try to follow it over time.
- Cost: usually **cheaper** than running a full detector every frame.

Tracking is useful when:
- You want **smooth trajectories** of objects.
- You want to **reduce computation** by not running detection on every frame.
- You want to maintain **identity** (ID) of each person over time.

#### 2.3 Classical Trackers: KCF and CSRT

OpenCV provides several classical trackers, including:

- **KCF (Kernelized Correlation Filter)**
  - Fast, works well when the object appearance does not change too much.
  - Sensitive to occlusions and large scale changes.

- **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
  - More accurate than KCF in many cases.
  - Slower than KCF, but still real-time on CPU for moderate resolutions.

These trackers are **not deep learning models**; they are classical computer vision algorithms.

#### 2.4 Tracking-by-Detection

**Tracking-by-detection** combines both ideas:

1. Run a **detector** every N frames (e.g., every 5 or 10 frames).
2. Initialize or update **trackers** for each detected object.
3. Between detection frames, use the trackers to **propagate** object positions.

Advantages:
- More robust than pure tracking (because detection can recover from drift).
- More efficient than running detection on every frame.
- Natural fit for systems that need **IDs over time**, like counting people entering a store or tracking their paths.

In the master’s thesis system, a similar idea is used: detect people, then classify age and gender for each detection, and aggregate statistics over time in a real deployment.

In [ ]:
video_path = 'race_car.mp4'  # Replace with your video path

In [ ]:
from IPython.display import HTML
HTML("""
<video width="640" height="480" controls>
    <source src="race_car.mp4">
</video>
""")

### **KCF (Kernelized Correlation Filter)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Track one object based only on its appearance from the first frame

In [ ]:
import cv2
from ultralytics import YOLO

# Cargar modelo
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

# Inicializar tracker
tracker = cv2.TrackerKCF_create()
tracker.init(frame, bbox)

# LOOP DE VISUALIZACIÓN
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, "Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("KCF Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

### **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Improved version of KCF that adapts to scale and appearance changes

In [ ]:
import cv2
from ultralytics import YOLO

# Cargar modelo YOLO
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:  # clase "car"
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box (xyxy → xywh)
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

print("BBox inicial:", bbox)

# 🔥 Inicializar CSRT (más robusto que KCF)
tracker = cv2.TrackerCSRT_create()

# (si falla, usa esta alternativa)
# tracker = cv2.legacy.TrackerCSRT_create()

tracker.init(frame, bbox)

# LOOP de tracking
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255,0,0), 2)
        cv2.putText(frame, "CSRT Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("CSRT Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

### **YOLO + ByteTrack**
- Type: **Multi-Object Tracking (MOT)**
- Requires: **Detection on every frame (YOLO)**
- Uses IoU + confidence score for association

**Key idea:** Fast and efficient tracking-by-detection without deep appearance features

In [ ]:
from ultralytics import YOLO

# Load an official or custom model
model = YOLO("yolo26n.pt")  # Load an official Detect model

#results = model.track(source="/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4", show=True)  # Tracking with default tracker - BoT-SORT
results = model.track("https://youtu.be/LNwODJXcvt4", show=True, tracker="bytetrack.yaml")  # with ByteTrack

### **Summary**

| Method | Type | Uses Detection Every Frame? | # Objects | Robustness |
|-------|------|----------------------------|----------|-----------|
| KCF | SOT | ❌ No | 1 | Low |
| CSRT | SOT | ❌ No | 1 | Medium |
| YOLO + ByteTrack | MOT | ✅ Yes | Many | High |

**One-Line Intuition**

- KCF → "Follow this object fast"
- CSRT → "Follow this object carefully"
- YOLO + ByteTrack → "Track many objects efficiently"

### **The Tracking Challenge: Motion Trail + Heatmap + Smart Counter**

- Work in **groups of 2 students**

#### Objective

Build a **multi-feature tracking system** using a video of people (the video will be provided by the instructor) that includes:

1. Motion Trail (trajectory visualization)
2. Heatmap of activity
3. Smart Entry Counter

#### Task Overview

You will use an object tracking method (e.g., YOLO + ByteTrack) to:

- Track people across frames
- Extract their positions over time
- Build three different visual/analytical outputs

#### **Task 1: Motion Trail (Trajectory)**

Requirements:
- For each tracked person:
  - Compute the **center point** of the bounding box
  - Draw a **trail (path)** of previous positions
- The trail must:
  - Fade over time (older points disappear gradually)
  - Be updated frame-by-frame

#### **Task 2: Heatmap of Activity**

Requirements:
- Accumulate all detected center points into a spatial grid
- Generate a **heatmap visualization** showing:
  - High-density areas (more traffic)
  - Low-density areas
- Overlay the heatmap on the video or display separately

### **Task 3: Smart Entry Counter**

- Define a **virtual line** (e.g., entrance)
- Count how many people:
  - Cross the line in a specific direction
- Avoid double counting:
  - Each tracked ID should be counted only once

### **Deliverables**

Each group must submit:

1. Working code
2. Video output
3. Short explanation (ONLY ONE of the following per student) of the working logic:
   - Motion Trail
   - Heatmap
   - Counter logic

### **Grading Rubric (10 points total)**

#### Functionality (6 points)

| Component | Points | Criteria |
|----------|--------|---------|
| Motion Trail | 2 pts | Works correctly (center + fading) OR not |
| Heatmap | 2 pts | Correct accumulation and visualization OR not |
| Counter | 2 pts | Correct counting (no duplicates) OR not |

#### Explanation (4 points)

| Criteria | Points |
|--------|--------|
| Clear and correct explanation of chosen component | 4 pts |
| Incorrect or unclear explanation | 0 pts |
- No partial credit

In [ ]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from collections import defaultdict

class TrackingSystem:
    """Multi-feature tracking system optimized for GPU"""
    
    def __init__(self, video_path, output_path="output_tracking.mp4", max_trail_length=30):
        self.video_path = video_path
        self.output_path = output_path
        self.max_trail_length = max_trail_length
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"✅ Using device: {self.device}")
        
        self.model = YOLO("yolov8n.pt")
        self.model.to(self.device)
        
        self.trails = defaultdict(list)
        self.heatmap = None
        self.heatmap_age = None
        self.current_frame = 0
        self.counted_ids = set()
        self.entry_count = 0
        self.exit_count = 0
        
        self.cap = cv2.VideoCapture(video_path)
        self.fps = int(self.cap.get(cv2.CAP_PROP_FPS))
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        self.heatmap = np.zeros((self.height, self.width), dtype=np.float32)
        self.heatmap_age = np.zeros((self.height, self.width), dtype=np.float32)
        
        self.line_y = self.height // 3
        self.line_thickness = 3
        
        print(f"Video info: {self.width}x{self.height} @ {self.fps} FPS")
        
    def get_center(self, box):
        x1, y1, x2, y2 = box
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        return (cx, cy)
    
    def update_trail(self, track_id, center):
        self.trails[track_id].append(center)
        if len(self.trails[track_id]) > self.max_trail_length:
            self.trails[track_id].pop(0)
    
    def draw_trail(self, frame, track_id):
        """Draw fading trail: red (recent) → blue (old)"""
        trail = self.trails[track_id]
        if len(trail) < 2:
            return
        
        for i in range(1, len(trail)):
            t = i / len(trail)  # 0.0=oldest, 1.0=newest
            
            r = int(255 * t)
            g = 0
            b = int(255 * (1 - t))
            
            alpha = t
            color = (int(b * alpha), int(g * alpha), int(r * alpha))  # BGR
            
            thickness = max(1, int(3 * t))
            cv2.line(frame, trail[i-1], trail[i], color, thickness)
        
        cv2.circle(frame, trail[-1], 5, (0, 0, 255), -1)
    
    def update_heatmap(self, center):
        """Acumula densidad en los puntos donde se detecta presencia"""
        x, y = center
        if 0 <= x < self.width and 0 <= y < self.height:
            # En lugar de poner un 1 fijo, sumamos un valor pequeño.
            # Esto crea "puntos calientes" donde la gente se queda quieta.
            # Usamos una máscara temporal para sumar solo en el área del círculo.
            temp_mask = np.zeros((self.height, self.width), dtype=np.float32)
            cv2.circle(temp_mask, center, 15, 0.05, -1) # 0.05 es la tasa de acumulación
            self.heatmap += temp_mask

    def draw_heatmap(self, frame, alpha=0.4):
        """Heatmap de Densidad: Azul (poca frecuencia) → Rojo (mucha frecuencia)"""
        max_val = np.max(self.heatmap)
        if max_val == 0:
            return frame
    
        # Normalizamos la densidad: 0 = nadie, 1 = zona de máxima concurrencia
        density_normalized = self.heatmap / max_val
        visited_mask = self.heatmap > 0
        
        t = density_normalized  # t ahora representa intensidad/densidad
    
        # Paleta JET aplicada a la densidad
        r = np.clip(255 * (1.5 * t - 0.5),   0, 255)
        g = np.clip(255 * np.where(t < 0.5,
                                   2 * t,
                                   -2 * t + 2), 0, 255)
        b = np.clip(255 * (1.0 - 1.5 * t),   0, 255)
    
        heatmap_bgr = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        heatmap_bgr[:, :, 0] = (b * visited_mask).astype(np.uint8)
        heatmap_bgr[:, :, 1] = (g * visited_mask).astype(np.uint8)
        heatmap_bgr[:, :, 2] = (r * visited_mask).astype(np.uint8)
    
        # Aplicamos un desenfoque (blur) para que el mapa de densidad se vea suave
        heatmap_bgr = cv2.GaussianBlur(heatmap_bgr, (15, 15), 0)
    
        overlay = frame.copy()
        # Solo aplicamos el color donde la densidad sea significativa
        overlay[visited_mask] = cv2.addWeighted(frame, 1 - alpha, heatmap_bgr, alpha, 0)[visited_mask]
    
        return overlay
    
    def cross_virtual_line(self, prev_center, curr_center):
        if prev_center is None:
            return None
        
        py, cy = prev_center[1], curr_center[1]
        
        if py < self.line_y <= cy:
            return "entering"
        elif py >= self.line_y > cy:
            return "exiting"
        return None
    
    def count_crossing(self, track_id, crossing_type):
        key = (track_id, crossing_type)
        if key not in self.counted_ids:
            self.counted_ids.add(key)
            if crossing_type == "entering":
                self.entry_count += 1
            else:
                self.exit_count += 1
            return True
        return False
    
    def process_video(self, show_heatmap=True, show_trail=True, show_counter=True):
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(self.output_path, fourcc, self.fps, 
                             (self.width, self.height))
        
        frame_idx = 0
        prev_centers = {}
        
        print("Processing video...")
        
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break
            
            frame_idx += 1
            self.current_frame = frame_idx  # 🆕 update global frame counter
            
            results = self.model.track(frame, persist=True, verbose=False, device=self.device, imgsz=320)
            
            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)
                classes = results[0].boxes.cls.cpu().numpy().astype(int)
                
                for box, track_id, cls in zip(boxes, ids, classes):
                    if cls != 0:
                        continue
                    
                    center = self.get_center(box)
                    
                    if show_trail:
                        self.update_trail(track_id, center)
                    
                    if show_heatmap:
                        self.update_heatmap(center)
                    
                    if show_counter:
                        prev_center = prev_centers.get(track_id)
                        crossing = self.cross_virtual_line(prev_center, center)
                        if crossing:
                            self.count_crossing(track_id, crossing)
                    
                    prev_centers[track_id] = center
            
            output_frame = frame.copy()
            
            if show_counter:
                cv2.line(output_frame, (0, self.line_y), (self.width, self.line_y), (255, 0, 0), self.line_thickness)
            
            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                ids = results[0].boxes.id.cpu().numpy().astype(int)
                classes = results[0].boxes.cls.cpu().numpy().astype(int)
                
                for box, track_id, cls in zip(boxes, ids, classes):
                    if cls != 0:
                        continue
                    
                    x1, y1, x2, y2 = map(int, box)
                    cv2.rectangle(output_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    if show_trail:
                        self.draw_trail(output_frame, track_id)
            
            if show_heatmap:
                output_frame = self.draw_heatmap(output_frame, alpha=0.3)
            
            if show_counter:
                cv2.putText(output_frame, f"Entries: {self.entry_count}", (10, 50),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(output_frame, f"Exits: {self.exit_count}", (10, 100),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            
            out.write(output_frame)
            
            if frame_idx % 30 == 0:
                print(f"Processed: {frame_idx}/{self.frame_count} frames")
        
        self.cap.release()
        out.release()
        
        print(f"\n✅ Video saved: {self.output_path}")
        print(f"📊 Statistics: Entries={self.entry_count}, Exits={self.exit_count}")


if __name__ == "__main__":
    tracker = TrackingSystem("store.mp4", output_path="output_gpu_fi.mp4")
    tracker.process_video(show_heatmap=True, show_trail=True, show_counter=True)

✅ Using device: cuda
Video info: 1280x720 @ 30 FPS
Processing video...
Processed: 30/10642 frames
Processed: 60/10642 frames
Processed: 90/10642 frames
Processed: 120/10642 frames
Processed: 150/10642 frames
Processed: 180/10642 frames
Processed: 210/10642 frames
Processed: 240/10642 frames
Processed: 270/10642 frames
Processed: 300/10642 frames
Processed: 330/10642 frames
Processed: 360/10642 frames
Processed: 390/10642 frames
Processed: 420/10642 frames
Processed: 450/10642 frames
Processed: 480/10642 frames
Processed: 510/10642 frames
Processed: 540/10642 frames
Processed: 570/10642 frames
Processed: 600/10642 frames
Processed: 630/10642 frames
Processed: 660/10642 frames
Processed: 690/10642 frames
Processed: 720/10642 frames
Processed: 750/10642 frames
Processed: 780/10642 frames
Processed: 810/10642 frames
Processed: 840/10642 frames
Processed: 870/10642 frames
Processed: 900/10642 frames
Processed: 930/10642 frames
Processed: 960/10642 frames
Processed: 990/10642 frames
Processe

---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>